# 🏦 Estimating Financial Needs & Next Best Action (NBA) Recommendation System
## Fully Explained ML / DL Notebook

This notebook is written so that **the code stays exactly as it is**, while the **markdown explains the logic clearly** from start to finish.

### What this notebook does
We solve the case study in a structured and defensible way:

1. **Read the dataset and supporting sheets**
   - client-level data
   - product catalogue
   - metadata / data dictionary

2. **Audit data quality**
   - inspect shape, types, duplicates, missing values, distributions, and basic sanity checks

3. **Explore the targets and features deeply**
   - class balance
   - univariate distributions
   - monetary-variable skewness
   - correlations
   - group differences and statistical tests
   - interactions between features and targets
   - PCA for structure and visualization

4. **Build a leakage-safe modelling pipeline**
   - shared train / validation / test split
   - feature engineering done carefully
   - scaling only where it is methodologically appropriate
   - multiple classical ML models benchmarked fairly

5. **Add deep learning**
   - train a PyTorch MLP under the same disciplined validation framework
   - compare it against the best tabular ML candidate

6. **Evaluate correctly**
   - cross-validation on training only
   - threshold tuning on validation
   - one final locked test evaluation

7. **Explain predictions**
   - permutation importance
   - SHAP
   - global explainability after final model choice

8. **Convert predictions into recommendations**
   - estimate needs
   - filter products by need type
   - apply risk suitability
   - rank final Next Best Action candidates

---

## How to read this notebook
A simple way to explain it in class is:

- **Section 1–2:** “First we understand the data.”
- **Section 3–4:** “Then we build and compare models in a leakage-safe way.”
- **Section 5–7:** “Then we stress-test, explain, and validate the final choices.”
- **Section 8–9:** “Finally we turn predictions into business recommendations and summarize the findings.”

---

## Why this version is strong
This notebook is not just trying to get a high metric. It is trying to be:
- **academically defendable**
- **methodologically clean**
- **business-relevant**
- **interpretable**
- **ambitious enough to stand out**

So while the code is advanced, the story is kept simple:
> **understand the data → build disciplined models → explain the results → generate recommendations**


In [ ]:
# ═══════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (run this cell once)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

packages = ['xgboost', 'lightgbm', 'shap', 'imbalanced-learn', 'xlrd']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All packages installed.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (chi2_contingency, mannwhitneyu, spearmanr,
                          pearsonr, ks_2samp, shapiro, kruskal,
                          pointbiserialr, normaltest)

# Sklearn
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      RandomizedSearchCV, cross_val_score,
                                      cross_val_predict, learning_curve)
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                    PowerTransformer, LabelEncoder)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Sklearn Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier,
                               BaggingClassifier, HistGradientBoostingClassifier,
                               StackingClassifier, VotingClassifier)
from sklearn.neural_network import MLPClassifier

# Sklearn Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report,
                              precision_recall_curve, average_precision_score,
                              matthews_corrcoef, balanced_accuracy_score,
                              log_loss)
from sklearn.inspection import permutation_importance

# XGBoost & LightGBM
import xgboost as xgb
import lightgbm as lgb

# SHAP
import shap

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

print("✅ All libraries loaded successfully.")
print(f"   PyTorch  : {torch.__version__}")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")
print(f"   SHAP     : {shap.__version__}")


---
# 1. Data Ingestion & Quality Audit

This section is the starting point of the entire pipeline.

## Goal of this section
Before doing any modelling, we need to answer basic questions:

- What files are available?
- What does each sheet contain?
- What do the columns mean?
- Is the data already clean, or are there quality issues?

## What the next code cells do
The next cells:
1. **load the Excel workbook**
2. **separate the sheets** into:
   - the main client dataset
   - the products catalogue
   - the metadata table
3. **inspect structure and variable meaning**
4. create a quick understanding of what each table will be used for later

## Why this matters
If we skip this step, we risk:
- using the wrong sheet for modelling
- misunderstanding a column
- creating features from the wrong variable
- building a model without knowing how targets and products relate

So this section is not “just loading data.”
It is where we define the raw material for everything that follows.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════
# ---- Adjust this path for your environment ----
# Google Colab with Drive:

# Local / uploaded file:
FILE_PATH = 'Dataset2_Needs.xls'  # <-- Put your .xls file in the same folder as this notebook

needs_df    = pd.read_excel(FILE_PATH, sheet_name='Needs')
products_df = pd.read_excel(FILE_PATH, sheet_name='Products')
metadata_df = pd.read_excel(FILE_PATH, sheet_name='Metadata')

# Fix trailing space in column name
needs_df.rename(columns={'Income ': 'Income'}, inplace=True)

print(f"Clients dataset : {needs_df.shape[0]:,} rows × {needs_df.shape[1]} columns")
print(f"Products catalog : {products_df.shape[0]} products × {products_df.shape[1]} attributes")
print(f"\nColumns: {needs_df.columns.tolist()}")
needs_df.head()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA DICTIONARY & PRODUCT CATALOG
# ═══════════════════════════════════════════════════════════════
meta_dict = dict(zip(
    metadata_df['Metadata'].dropna().astype(str),
    metadata_df['Unnamed: 1'].dropna().astype(str)
))
print("DATA DICTIONARY:")
for var, desc in meta_dict.items():
    print(f"  {str(var):30s} → {desc}")

product_names = {
    1: 'Balanced Mutual Fund', 2: 'Income Conservative Unit-Linked',
    3: 'Fixed Income Mutual Fund', 4: 'Balanced High Dividend MF',
    5: 'Balanced Mutual Fund II', 6: 'Defensive Flex Unit-Linked',
    7: 'Aggressive Flex Unit-Linked', 8: 'Balanced Flex Unit-Linked',
    9: 'Cautious Alloc Segregated', 10: 'Fixed Income Segregated',
    11: 'Total Return Aggr Segregated'
}
products_df['Name'] = products_df['IDProduct'].map(product_names)
products_df['TypeLabel'] = products_df['Type'].map({1: 'Accumulation', 0: 'Income'})
print("\nPRODUCT CATALOG:")
print(products_df.to_string(index=False))


---
# 2. Exploratory Data Analysis (EDA)

## Why we do EDA here
Before training models, we need to understand:
- how the variables behave
- whether the targets are balanced or imbalanced
- whether monetary features are skewed
- whether different client groups behave differently
- whether relationships are linear, weak, strong, or noisy

In short, EDA tells us **what kind of modelling problem we are dealing with**.

## What this section covers
This EDA is intentionally richer than a basic classroom notebook. It includes:
- data quality and descriptive statistics
- target class balance
- distribution diagnostics
- transformations for fat-tailed monetary variables
- correlation analysis
- formal statistical testing
- interaction analysis
- dimensionality reduction via PCA

## How to explain this section in presentation terms
You can say:

> “We did deep EDA not just for visuals, but to justify our later modelling choices — especially feature engineering, scaling policy, and model family selection.”

## 2.1 Descriptive Statistics & Data Quality
The next code block builds a general diagnostic picture of the dataset:
- size
- data types
- missingness
- duplicates
- summary statistics
- initial sanity checks


In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE DATA QUALITY REPORT
# ═══════════════════════════════════════════════════════════════
df = needs_df.drop('ID', axis=1).copy()

print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)
print(f"  Total observations      : {len(df):,}")
print(f"  Missing values (total)  : {df.isnull().sum().sum()}")
print(f"  Duplicate rows          : {df.duplicated().sum()}")
print(f"  Memory usage            : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS (EXTENDED)")
print("=" * 70)
desc = df.describe().T
desc['skewness'] = df.skew()
desc['kurtosis'] = df.kurtosis()
desc['IQR'] = desc['75%'] - desc['25%']
desc['CV'] = desc['std'] / desc['mean']  # Coefficient of variation
print(desc.round(4).to_string())

# Data types
print("\nData Types:")
print(df.dtypes.to_string())


## 2.2 Target Variable Analysis — Class Balance

Now we focus specifically on the two targets:
- `IncomeInvestment`
- `AccumulationInvestment`

## What the next code does
It checks:
- how many positive vs negative cases we have for each target
- whether one target is more imbalanced than the other
- whether the targets are statistically associated

## Why this matters
Target balance affects:
- which metrics are important
- how confident we should be in raw accuracy
- whether threshold tuning becomes important
- how difficult the classification task is

If classes are uneven, a model can look “good” on accuracy while actually missing the minority class.
That is why class balance analysis comes early.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# TARGET CLASS BALANCE + CHI-SQUARED INDEPENDENCE TEST
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes[:2], ['IncomeInvestment', 'AccumulationInvestment']):
    counts = df[target].value_counts()
    ratio = counts[1] / len(df)
    colors = ['#3498db', '#e74c3c']
    bars = ax.bar(['No Need (0)', 'Has Need (1)'], counts.values,
                   color=colors, edgecolor='black', alpha=0.85)
    ax.set_title(f'{target}\nPrevalence: {ratio:.1%} | Imbalance ratio: {counts[0]/counts[1]:.2f}:1',
                 fontsize=11)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 30, str(v),
                ha='center', fontweight='bold')

# Cross-tabulation heatmap
ct = pd.crosstab(df['IncomeInvestment'], df['AccumulationInvestment'])
chi2, p, dof, expected = chi2_contingency(ct)
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[2])
axes[2].set_title(f'Target Cross-Tab\nχ²={chi2:.2f}, p={p:.4f}')
axes[2].set_xlabel('AccumulationInvestment')
axes[2].set_ylabel('IncomeInvestment')

plt.tight_layout()
plt.show()

print(f"Chi-squared independence test: χ²={chi2:.2f}, p={p:.4e}")
print(f"→ Targets are {'associated' if p < 0.05 else 'independent'} (α=0.05)")
print(f"→ Separate binary classifiers (One-vs-All) are appropriate.")


## 2.3 Feature Distributions — Histograms, Box Plots, Q-Q Plots

This subsection studies how each feature behaves individually.

## What the next code does
For the main numeric features, it draws:
- **histograms** to see shape
- **box plots** to detect spread and outliers
- **Q-Q plots** to compare against a normal distribution

## Why we care
These plots help us answer questions like:
- Is a variable approximately normal?
- Is it heavily skewed?
- Are there extreme values?
- Would a log transformation make sense?
- Should we prefer robust or non-parametric statistics?

This becomes especially important for financial variables, which often have long right tails and non-normal behaviour.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DISTRIBUTION ANALYSIS: Histograms + Box Plots + Q-Q Plots
# ═══════════════════════════════════════════════════════════════
continuous_features = ['Age', 'FamilyMembers', 'FinancialEducation',
                       'RiskPropensity', 'Income', 'Wealth']

fig, axes = plt.subplots(len(continuous_features), 3, figsize=(18, 4*len(continuous_features)))

for i, feat in enumerate(continuous_features):
    # Histogram + KDE
    ax = axes[i, 0]
    sns.histplot(df[feat], kde=True, ax=ax, color='steelblue', bins=40, alpha=0.7)
    ax.axvline(df[feat].mean(), color='red', ls='--', lw=1.2, label=f'μ={df[feat].mean():.2f}')
    ax.axvline(df[feat].median(), color='green', ls=':', lw=1.2, label=f'med={df[feat].median():.2f}')
    ax.legend(fontsize=8)
    ax.set_title(f'{feat} — Distribution (skew={df[feat].skew():.2f})')

    # Box Plot
    ax2 = axes[i, 1]
    sns.boxplot(x=df[feat], ax=ax2, color='lightcoral')
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)).sum()
    ax2.set_title(f'{feat} — Box Plot (outliers: {outliers})')

    # Q-Q Plot
    ax3 = axes[i, 2]
    stats.probplot(df[feat], dist='norm', plot=ax3)
    ax3.set_title(f'{feat} — Q-Q Plot')

plt.tight_layout()
plt.show()

# Normality tests
print("\nNormality Tests (Shapiro-Wilk, subsample n=500):")
print("-" * 60)
for feat in continuous_features:
    sample = df[feat].sample(500, random_state=42)
    w_stat, w_p = shapiro(sample)
    k2, k_p = normaltest(sample)
    print(f"  {feat:25s}: Shapiro W={w_stat:.4f} (p={w_p:.2e}) | "
          f"D'Agostino K²={k2:.2f} (p={k_p:.2e}) → "
          f"{'Normal' if w_p > 0.05 and k_p > 0.05 else 'Non-normal'}")


## 2.4 Monetary Variable Transformations — Handling Fat Tails

Income and wealth variables are usually not nicely bell-shaped.  
They often have:
- strong skewness
- large outliers
- fat tails

## What the next code does
It compares raw and transformed versions of the monetary variables so we can judge whether:
- log scaling improves shape
- distributions become more stable
- modelling may become easier or more interpretable

## Why this matters
A transformation is not done “because it looks fancy.”
It is done because:
- some models behave better with less skewed inputs
- effect sizes become more stable
- visual and statistical interpretation improves

So this step gives a principled reason for later engineered features such as log-income and log-wealth style variables.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# WEALTH & INCOME TRANSFORMATION COMPARISON
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for row, feat in enumerate(['Wealth', 'Income']):
    data = df[feat]

    # Raw
    sns.histplot(data, kde=True, ax=axes[row, 0], color='steelblue', bins=50)
    axes[row, 0].set_title(f'{feat} — Raw\nskew={data.skew():.2f}, kurt={data.kurtosis():.2f}')

    # Log1p
    log_data = np.log1p(data)
    sns.histplot(log_data, kde=True, ax=axes[row, 1], color='darkorange', bins=50)
    axes[row, 1].set_title(f'{feat} — log1p\nskew={log_data.skew():.2f}')

    # Yeo-Johnson
    pt = PowerTransformer(method='yeo-johnson')
    yj_data = pt.fit_transform(data.values.reshape(-1, 1)).flatten()
    sns.histplot(yj_data, kde=True, ax=axes[row, 2], color='forestgreen', bins=50)
    axes[row, 2].set_title(f'{feat} — Yeo-Johnson\nskew={pd.Series(yj_data).skew():.2f}')

plt.suptitle('Transformation Comparison for Fat-Tailed Monetary Variables', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("→ log1p provides good normalization; Yeo-Johnson achieves near-zero skewness.")
print("→ We will use log1p in feature engineering (interpretable) and StandardScaler for model input.")


## 2.5 Correlation Analysis & Statistical Hypothesis Testing

This subsection moves from **visual inspection** to **formal evidence**.

## What the next code blocks do
They measure relationships using:
- Pearson / Spearman style correlations
- point-biserial relationships with binary targets
- group-comparison tests such as Mann–Whitney U
- association analysis between categorical variables

## Why this matters
This helps us answer:
- Which features appear related to the targets?
- Are those relationships linear or monotonic?
- Are observed group differences statistically meaningful?
- Which variables deserve more modelling attention?

This is useful both academically and practically:
- academically, because it shows statistical grounding
- practically, because it helps us decide which features and transformations matter most


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CORRELATION HEATMAPS (PEARSON + SPEARMAN)
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mask = np.triu(np.ones_like(df.corr(), dtype=bool))

sns.heatmap(df.corr(method='pearson'), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], vmin=-1, vmax=1, mask=mask, square=True)
axes[0].set_title('Pearson Correlation', fontsize=12)

sns.heatmap(df.corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], vmin=-1, vmax=1, mask=mask, square=True)
axes[1].set_title('Spearman Rank Correlation', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# POINT-BISERIAL CORRELATIONS + MANN-WHITNEY U TESTS
# ═══════════════════════════════════════════════════════════════
features_list = ['Age', 'Gender', 'FamilyMembers', 'FinancialEducation',
                 'RiskPropensity', 'Income', 'Wealth']

print("=" * 80)
print("POINT-BISERIAL CORRELATIONS WITH TARGETS")
print("=" * 80)
for target in ['IncomeInvestment', 'AccumulationInvestment']:
    print(f"\n  {target}:")
    for feat in features_list:
        r, p = pointbiserialr(df[target], df[feat])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"    {feat:25s}: r={r:+.4f}, p={p:.4e} {sig}")

print("\n" + "=" * 80)
print("MANN-WHITNEY U TESTS (Y=0 vs Y=1)")
print("=" * 80)
for target in ['IncomeInvestment', 'AccumulationInvestment']:
    print(f"\n  {target}:")
    for feat in features_list:
        g0 = df.loc[df[target] == 0, feat]
        g1 = df.loc[df[target] == 1, feat]
        u_stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
        eff_r = 1 - (2*u_stat) / (len(g0)*len(g1))  # rank-biserial
        print(f"    {feat:25s}: U={u_stat:>12,.0f}, p={p:.4e}, r_rb={eff_r:+.3f} "
              f"{'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'}")


## 2.6 Feature Interactions — Violin Plots by Target

A single variable on its own does not always tell the full story.  
Sometimes the **distribution of a feature changes meaningfully by target class**.

## What the next code does
It visualizes how important variables are distributed for:
- clients with the need
- clients without the need

using interaction-friendly plots such as violin plots and grouped comparisons.

## Why this matters
This helps reveal:
- overlap vs separation between classes
- whether class differences are small or large
- whether a feature may help classification
- whether nonlinear or interaction-aware models could be valuable

This is one of the bridges from EDA into model design.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VIOLIN PLOTS — FEATURE DISTRIBUTIONS BY TARGET
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for row, target in enumerate(['IncomeInvestment', 'AccumulationInvestment']):
    for col, feat in enumerate(['Age', 'RiskPropensity', 'Income', 'Wealth']):
        ax = axes[row, col]
        sns.violinplot(x=df[target], y=df[feat], ax=ax,
                       palette=['#3498db', '#e74c3c'], inner='quartile',
                       cut=0, bw_adjust=0.8)
        # Overlay strip plot for granularity
        sns.stripplot(x=df[target], y=df[feat], ax=ax,
                      color='black', alpha=0.05, size=1)
        ax.set_title(f'{feat} by {target}', fontsize=10)
        ax.set_xlabel('')

plt.suptitle('Feature Distributions Segmented by Investment Need', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENDER × TARGET INTERACTION ANALYSIS
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['IncomeInvestment', 'AccumulationInvestment']):
    ct = pd.crosstab(df['Gender'], df[target], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'],
            edgecolor='black', alpha=0.85)
    ax.set_title(f'Need Prevalence by Gender — {target}')
    ax.set_xlabel('Gender (0=Male, 1=Female)')
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(['Male', 'Female'], rotation=0)
    ax.legend(['No Need', 'Has Need'])

plt.tight_layout()
plt.show()

# Kruskal-Wallis test: do family members differ by combined target group?
df['NeedGroup'] = df['IncomeInvestment'].astype(str) + '_' + df['AccumulationInvestment'].astype(str)
groups = [g['FamilyMembers'].values for _, g in df.groupby('NeedGroup')]
h_stat, h_p = kruskal(*groups)
print(f"Kruskal-Wallis test (FamilyMembers across need groups): H={h_stat:.2f}, p={h_p:.4e}")
df.drop('NeedGroup', axis=1, inplace=True)


## 2.7 Dimensionality Reduction — PCA Scree Plot & 2D Projections

After univariate and pairwise exploration, we zoom out and ask:

> Is there any broader structure in the feature space?

## What the next code does
It applies PCA to:
- inspect explained variance
- understand how much information lives in a few components
- visualize the dataset in reduced dimensions

## Why this is useful
PCA here is mainly an **exploratory tool**, not the final modelling strategy.

It helps us:
- check whether information is concentrated or diffuse
- visualize broad patterns
- understand redundancy in the features
- support discussion of complexity and separability

So this is not “PCA for the sake of PCA.”  
It is a diagnostic step that enriches our understanding of the feature space.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PCA ANALYSIS + SCREE PLOT
# ═══════════════════════════════════════════════════════════════
X_viz = df[features_list].copy()
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X_viz)

# Full PCA
pca_full = PCA(random_state=42).fit(X_scaled_viz)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Scree plot
ax = axes[0]
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
ax.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
       pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue', label='Individual')
ax.plot(range(1, len(cumvar)+1), cumvar, 'ro-', label='Cumulative')
ax.axhline(y=0.95, color='gray', ls='--', alpha=0.5, label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Scree Plot')
ax.legend(fontsize=9)

# 2D PCA colored by each target
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled_viz)

for ax, target in zip(axes[1:], ['IncomeInvestment', 'AccumulationInvestment']):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df[target],
                         cmap='RdYlBu', alpha=0.35, s=8, edgecolors='none')
    ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
    ax.set_title(f'PCA — {target}')
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.show()

print(f"PCA explained variance (2 components): {pca_2d.explained_variance_ratio_.sum():.1%}")
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 95% variance: {n_95}")


---
# 3. Feature Engineering, Modeling & Evaluation Pipeline

We now move from **understanding the data** to **building the predictive system**.

## Design philosophy of this pipeline
This notebook follows a disciplined order:

1. **Create one shared split**
   - same train / validation / test partition for both targets

2. **Engineer features safely**
   - no leakage from validation/test into training logic

3. **Scale only when needed**
   - not every model should receive the same preprocessing

4. **Benchmark models fairly**
   - cross-validation only on training data

5. **Tune thresholds on validation**
   - not on the test set

6. **Use the test set once**
   - final reporting only

## Why this is important
A flashy notebook can still be weak if the evaluation design is sloppy.  
This section is where the notebook becomes **defendable under scrutiny**, because the modelling workflow is separated cleanly into:
- training
- validation
- final test

That separation is one of the strongest parts of the whole project.


In [ ]:
from sklearn.model_selection import cross_validate
from pathlib import Path

# Additional imports for the upgraded final pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import binomtest
from IPython.display import display

OUTPUT_DIR = Path("merged_nba_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("Upgrade patch loaded. Outputs will be saved to:", OUTPUT_DIR.resolve())


## 3.1 Leakage-Safe Feature Engineering & Train / Validation / Test Split

This is one of the most important technical sections in the notebook.

## What the next code does
It performs three big tasks:

### A. Create engineered features
The notebook generates richer predictors from the raw inputs so the models can capture:
- nonlinearity
- financial scale effects
- interaction structure
- life-cycle style client patterns

### B. Build one shared split
Instead of splitting separately for each target, the notebook creates a **single global split** that is reused consistently.

### C. Protect against leakage
Feature engineering decisions are made carefully so we do not accidentally use information from validation or test data while preparing training inputs.

## Why this matters
Leakage is one of the easiest ways to get fake performance.
A model may look excellent, but only because it has indirectly “seen” information it should never have seen.

So this step is not just preprocessing — it is the part that keeps the benchmark honest.


In [ ]:

# Work from the cleaned client table
df_final = needs_df.copy()

base_features_v2 = ['Age', 'Gender', 'FamilyMembers', 'FinancialEducation',
                    'RiskPropensity', 'Income', 'Wealth']
target_cols_v2 = ['IncomeInvestment', 'AccumulationInvestment']

def engineer_features_v2(df_in):
    X = df_in[base_features_v2].copy()

    # Raw variables
    X['LogIncome'] = np.log1p(X['Income'].clip(lower=0))
    X['LogWealth'] = np.log1p(X['Wealth'].clip(lower=0))

    # Ratios / household-adjusted metrics
    X['IncomePerFamilyMember'] = X['Income'] / (X['FamilyMembers'] + 1.0)
    X['WealthPerFamilyMember'] = X['Wealth'] / (X['FamilyMembers'] + 1.0)
    X['WealthIncomeRatio'] = X['Wealth'] / (X['Income'] + 1.0)

    # Interactions
    X['RiskEducationInteraction'] = X['RiskPropensity'] * X['FinancialEducation']
    X['RiskWealthInteraction'] = X['RiskPropensity'] * X['LogWealth']
    X['AgeRiskInteraction'] = X['Age'] * X['RiskPropensity']

    # Nonlinearity
    X['AgeSquared'] = X['Age'] ** 2

    # Life-stage flags with fixed cutoffs (not learned from whole-dataset quantiles)
    X['Age_Under35'] = (X['Age'] < 35).astype(int)
    X['Age_35_54'] = ((X['Age'] >= 35) & (X['Age'] < 55)).astype(int)
    X['Age_55_69'] = ((X['Age'] >= 55) & (X['Age'] < 70)).astype(int)
    X['Age_70plus'] = (X['Age'] >= 70).astype(int)

    return X

X_base_v2 = df_final[base_features_v2].copy()
X_eng_v2 = engineer_features_v2(df_final)

y_income_v2 = df_final['IncomeInvestment'].astype(int)
y_accum_v2 = df_final['AccumulationInvestment'].astype(int)

joint_strat_v2 = y_income_v2.astype(str) + "_" + y_accum_v2.astype(str)
all_idx_v2 = np.arange(len(df_final))

train_val_idx_v2, test_idx_v2 = train_test_split(
    all_idx_v2,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=joint_strat_v2
)

train_idx_v2, val_idx_v2 = train_test_split(
    train_val_idx_v2,
    test_size=0.25,  # 0.25 of 80% -> 20% total
    random_state=RANDOM_STATE,
    stratify=joint_strat_v2.iloc[train_val_idx_v2]
)

train_idx_v2 = np.sort(train_idx_v2)
val_idx_v2 = np.sort(val_idx_v2)
test_idx_v2 = np.sort(test_idx_v2)
train_val_idx_v2 = np.sort(train_val_idx_v2)

split_summary_v2 = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n_rows': [len(train_idx_v2), len(val_idx_v2), len(test_idx_v2)],
    'income_positive_rate': [
        y_income_v2.iloc[train_idx_v2].mean(),
        y_income_v2.iloc[val_idx_v2].mean(),
        y_income_v2.iloc[test_idx_v2].mean(),
    ],
    'accum_positive_rate': [
        y_accum_v2.iloc[train_idx_v2].mean(),
        y_accum_v2.iloc[val_idx_v2].mean(),
        y_accum_v2.iloc[test_idx_v2].mean(),
    ],
})
display(split_summary_v2)


## 3.2 Consolidated Statistical Testing

This section keeps the notebook tied to statistical reasoning rather than pure black-box modelling.

## What the next code does
It gathers together the main statistical checks used to support interpretation:
- Mann–Whitney U for distribution differences
- effect-size style summaries
- chi-square tests
- Cramér’s V for association strength

## Why include this in a machine learning notebook?
Because in a strong academic project, we do not want to say only:

> “The model found something.”

We also want to say:

> “The data itself shows meaningful patterns, and the model is building on those patterns.”

So this section helps connect:
- classical statistics
- exploratory evidence
- modern ML modelling

That makes the whole project easier to defend in an oral explanation.


In [ ]:

def cramers_v_from_table(table):
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    denom = n * max(min(k - 1, r - 1), 1)
    return np.sqrt(chi2 / denom) if denom > 0 else np.nan

numeric_test_features_v2 = ['Age', 'FamilyMembers', 'FinancialEducation', 'RiskPropensity', 'Income', 'Wealth']
categorical_test_features_v2 = ['Gender']

def run_stat_tests_v2(df_in, target_col):
    rows = []
    for col in numeric_test_features_v2:
        g0 = df_in.loc[df_in[target_col] == 0, col]
        g1 = df_in.loc[df_in[target_col] == 1, col]
        mw_stat, mw_p = mannwhitneyu(g0, g1, alternative='two-sided')
        pb_r, pb_p = pointbiserialr(df_in[target_col], df_in[col])
        sp_r, sp_p = spearmanr(df_in[col], df_in[target_col])
        rows.append({
            'feature': col,
            'test': 'Mann-Whitney U',
            'group0_mean': g0.mean(),
            'group1_mean': g1.mean(),
            'group0_median': g0.median(),
            'group1_median': g1.median(),
            'pointbiserial_r': pb_r,
            'spearman_rho': sp_r,
            'mw_p_value': mw_p,
            'pb_p_value': pb_p,
            'sp_p_value': sp_p,
        })
    for col in categorical_test_features_v2:
        ct = pd.crosstab(df_in[col], df_in[target_col])
        chi2, p, dof, expected = chi2_contingency(ct)
        rows.append({
            'feature': col,
            'test': 'Chi-square',
            'group0_mean': np.nan,
            'group1_mean': np.nan,
            'group0_median': np.nan,
            'group1_median': np.nan,
            'pointbiserial_r': np.nan,
            'spearman_rho': np.nan,
            'mw_p_value': np.nan,
            'pb_p_value': np.nan,
            'sp_p_value': p,
            'cramers_v': cramers_v_from_table(ct),
        })
    out = pd.DataFrame(rows)
    return out

stats_income_v2 = run_stat_tests_v2(df_final, 'IncomeInvestment')
stats_accum_v2 = run_stat_tests_v2(df_final, 'AccumulationInvestment')

display(stats_income_v2)
display(stats_accum_v2)

stats_income_v2.to_csv(OUTPUT_DIR / 'stat_tests_income_v2.csv', index=False)
stats_accum_v2.to_csv(OUTPUT_DIR / 'stat_tests_accumulation_v2.csv', index=False)


## 4.1 Model Catalogue — Scaling Only Where Justified

Now the notebook defines the full model family that will be benchmarked.

## Why this section matters
Different algorithms have different preprocessing needs.

### Models that usually need scaled inputs
- Logistic Regression
- SVM
- sklearn MLP
- PyTorch MLP

### Models that usually do not need scaling
- Random Forest
- Extra Trees
- HistGradientBoosting
- XGBoost
- LightGBM

## What the next code does
It formalizes:
- which models are tested
- which feature sets they will use
- which scaling policy applies to each one

## Why this is a good methodological choice
A weak notebook often scales everything blindly.  
A stronger notebook scales **selectively and intentionally**.

That makes the pipeline:
- more principled
- more realistic
- easier to explain to a professor


In [ ]:

feature_sets_v2 = {
    'base': X_base_v2,
    'engineered': X_eng_v2,
}

scaled_model_names_v2 = {'Logistic Regression', 'SVM (RBF)', 'MLP (sklearn)'}

def build_model_catalog_v2(y_train):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = max(neg / max(pos, 1), 1.0)

    return {
        'Logistic Regression': LogisticRegression(max_iter=4000, class_weight='balanced', random_state=RANDOM_STATE),
        'SVM (RBF)': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(
            n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        'Extra Trees': ExtraTreesClassifier(
            n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        'HistGradientBoosting': HistGradientBoostingClassifier(
            max_iter=300, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, objective='binary:logistic', eval_metric='logloss',
            scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=4
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.03, num_leaves=31, subsample=0.9, colsample_bytree=0.9,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=4, verbose=-1
        ),
        'MLP (sklearn)': MLPClassifier(
            hidden_layer_sizes=(64, 32, 16), activation='relu',
            alpha=1e-3, learning_rate_init=0.003, batch_size=64,
            max_iter=500, early_stopping=True, validation_fraction=0.10,
            random_state=RANDOM_STATE
        ),
    }

def build_pipeline_v2(model_name, model):
    if model_name in scaled_model_names_v2:
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', model),
        ])
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', model),
    ])

def prediction_metrics_v2(y_true, proba, threshold=0.50):
    pred = (np.asarray(proba) >= threshold).astype(int)
    y_true = np.asarray(y_true)
    return {
        'threshold': threshold,
        'roc_auc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'accuracy': accuracy_score(y_true, pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, pred),
    }

def find_best_threshold_v2(y_true, proba, low=0.05, high=0.95, n_grid=181):
    grid = np.linspace(low, high, n_grid)
    rows = []
    for thr in grid:
        rows.append(prediction_metrics_v2(y_true, proba, threshold=thr))
    thr_df = pd.DataFrame(rows).sort_values(
        ['f1', 'pr_auc', 'balanced_accuracy', 'recall'],
        ascending=False
    ).reset_index(drop=True)
    return float(thr_df.iloc[0]['threshold']), thr_df


## 4.2 Validation-Driven Benchmarking & Model Selection

This is the core benchmarking engine of the notebook.

## What the next code does
It benchmarks models in a disciplined sequence:

1. **fit and cross-validate on training data**
2. **compare candidate configurations**
3. **use validation data to tune thresholds and choose the best option**
4. **do not touch the test set yet**

## Why this structure is strong
This avoids a common mistake:
- using the test set too early
- indirectly optimizing to the test set
- overestimating real performance

## What to say in simple words
You can explain this section like this:

> “We let the training data teach the models, we let the validation data choose the winner, and we save the test data for the final unbiased check.”

That one sentence summarizes the evaluation discipline of the entire notebook.


In [ ]:

def benchmark_feature_set_v2(feature_frame, y_series, target_name):
    X_train = feature_frame.iloc[train_idx_v2].copy()
    y_train = y_series.iloc[train_idx_v2].copy()
    X_val = feature_frame.iloc[val_idx_v2].copy()
    y_val = y_series.iloc[val_idx_v2].copy()

    model_catalog = build_model_catalog_v2(y_train)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    rows = []
    fitted_models = {}

    for model_name, model in model_catalog.items():
        pipe = build_pipeline_v2(model_name, clone(model))

        cv_scores = cross_validate(
            pipe,
            X_train, y_train,
            cv=cv,
            scoring={
                'roc_auc': 'roc_auc',
                'pr_auc': 'average_precision',
                'f1': 'f1',
                'balanced_accuracy': 'balanced_accuracy',
            },
            n_jobs=-1,
            return_train_score=False,
        )

        pipe.fit(X_train, y_train)
        val_proba = pipe.predict_proba(X_val)[:, 1]
        best_thr, thr_df = find_best_threshold_v2(y_val, val_proba)
        val_metrics = prediction_metrics_v2(y_val, val_proba, threshold=best_thr)

        fitted_models[model_name] = {
            'pipeline': pipe,
            'val_proba': val_proba,
            'threshold_grid': thr_df,
            'best_threshold': best_thr,
        }

        rows.append({
            'target': target_name,
            'model_name': model_name,
            'cv_roc_auc_mean': np.mean(cv_scores['test_roc_auc']),
            'cv_pr_auc_mean': np.mean(cv_scores['test_pr_auc']),
            'cv_f1_mean': np.mean(cv_scores['test_f1']),
            'cv_bal_acc_mean': np.mean(cv_scores['test_balanced_accuracy']),
            'val_threshold': best_thr,
            'val_roc_auc': val_metrics['roc_auc'],
            'val_pr_auc': val_metrics['pr_auc'],
            'val_f1': val_metrics['f1'],
            'val_balanced_accuracy': val_metrics['balanced_accuracy'],
            'val_precision': val_metrics['precision'],
            'val_recall': val_metrics['recall'],
            'val_mcc': val_metrics['mcc'],
        })

    out = pd.DataFrame(rows).sort_values(
        ['val_pr_auc', 'val_f1', 'cv_pr_auc_mean', 'cv_roc_auc_mean'],
        ascending=False
    ).reset_index(drop=True)

    return out, fitted_models

def benchmark_all_v2(feature_sets_dict, y_series, target_name):
    results = []
    fitted = {}
    for fname, frame in feature_sets_dict.items():
        res, fit = benchmark_feature_set_v2(frame, y_series, target_name)
        res.insert(1, 'feature_set', fname)
        results.append(res)
        fitted[fname] = fit
    final = pd.concat(results, ignore_index=True).sort_values(
        ['val_pr_auc', 'val_f1', 'cv_pr_auc_mean', 'cv_roc_auc_mean'],
        ascending=False
    ).reset_index(drop=True)
    return final, fitted

benchmark_income_v2, fitted_income_v2 = benchmark_all_v2(feature_sets_v2, y_income_v2, 'IncomeInvestment')
benchmark_accum_v2, fitted_accum_v2 = benchmark_all_v2(feature_sets_v2, y_accum_v2, 'AccumulationInvestment')

display(benchmark_income_v2.head(12))
display(benchmark_accum_v2.head(12))

best_income_cfg_v2 = benchmark_income_v2.iloc[0].to_dict()
best_accum_cfg_v2 = benchmark_accum_v2.iloc[0].to_dict()

best_cfg_v2 = pd.DataFrame([best_income_cfg_v2, best_accum_cfg_v2])
display(best_cfg_v2)

benchmark_income_v2.to_csv(OUTPUT_DIR / 'benchmark_income_v2.csv', index=False)
benchmark_accum_v2.to_csv(OUTPUT_DIR / 'benchmark_accumulation_v2.csv', index=False)


## 5. Deep Learning — PyTorch MLP Under Disciplined Validation

This is where the notebook adds the deep learning layer.

## Why include deep learning here?
The project is not just trying to say “we used PyTorch.”  
It is trying to answer a better question:

> “Does a disciplined neural-network approach actually improve the problem compared with strong tabular baselines?”

## What the next code does
It builds a PyTorch MLP classifier and evaluates it under the same logic as the tabular models:
- training is separated properly
- validation is used for model comparison
- performance is not reported carelessly from the training data

## Why this is important
Deep learning only adds value if it is compared fairly.  
Otherwise, it becomes decoration.

So this section is valuable because the notebook treats the PyTorch model as a **real competitor**, not as a marketing add-on.


In [ ]:

class TorchMLPBinaryClassifierV2:
    def __init__(self, hidden=(128, 64, 32), dropout=0.15, lr=1e-3, batch_size=128, max_epochs=120, patience=12, random_state=RANDOM_STATE):
        self.hidden = hidden
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.random_state = random_state

    def _prep(self, X, fit=False):
        X_df = pd.DataFrame(X).copy()
        if fit:
            self.imputer_ = SimpleImputer(strategy='median')
            X_imp = self.imputer_.fit_transform(X_df)
            self.scaler_ = StandardScaler()
            X_scaled = self.scaler_.fit_transform(X_imp)
        else:
            X_imp = self.imputer_.transform(X_df)
            X_scaled = self.scaler_.transform(X_imp)
        return X_scaled.astype('float32')

    def _build_net(self, input_dim):
        layers = []
        prev = input_dim
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(self.dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        return nn.Sequential(*layers)

    def fit(self, X_train, y_train, X_val, y_val):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        Xtr = self._prep(X_train, fit=True)
        Xv = self._prep(X_val, fit=False)
        ytr = np.asarray(y_train).astype('float32').reshape(-1, 1)
        yv = np.asarray(y_val).astype('float32').reshape(-1, 1)

        self.device_ = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model_ = self._build_net(Xtr.shape[1]).to(self.device_)

        pos = float((ytr == 1).sum())
        neg = float((ytr == 0).sum())
        pos_weight = torch.tensor([max(neg / max(pos, 1.0), 1.0)], dtype=torch.float32, device=self.device_)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=1e-4)

        train_loader = DataLoader(
            TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)),
            batch_size=self.batch_size,
            shuffle=True
        )

        Xv_t = torch.tensor(Xv, dtype=torch.float32, device=self.device_)
        best_state = None
        best_pr = -np.inf
        bad_epochs = 0
        self.history_ = []

        for epoch in range(self.max_epochs):
            self.model_.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(self.device_)
                yb = yb.to(self.device_)
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item())

            self.model_.eval()
            with torch.no_grad():
                val_proba = torch.sigmoid(self.model_(Xv_t)).cpu().numpy().reshape(-1)

            val_pr = average_precision_score(y_val, val_proba)
            val_f1 = f1_score(y_val, (val_proba >= 0.5).astype(int))
            self.history_.append({
                'epoch': epoch + 1,
                'train_loss': running_loss / max(len(train_loader), 1),
                'val_pr_auc': val_pr,
                'val_f1_at_0_5': val_f1,
            })

            if val_pr > best_pr:
                best_pr = val_pr
                bad_epochs = 0
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)

        self.best_val_pr_auc_ = best_pr
        return self

    def predict_proba(self, X):
        Xp = self._prep(X, fit=False)
        Xt = torch.tensor(Xp, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            proba = torch.sigmoid(self.model_(Xt)).cpu().numpy().reshape(-1)
        return np.vstack([1 - proba, proba]).T

def evaluate_pytorch_v2(feature_frame, y_series, target_name):
    X_train = feature_frame.iloc[train_idx_v2].copy()
    y_train = y_series.iloc[train_idx_v2].copy()
    X_val = feature_frame.iloc[val_idx_v2].copy()
    y_val = y_series.iloc[val_idx_v2].copy()

    torch_model = TorchMLPBinaryClassifierV2()
    torch_model.fit(X_train, y_train, X_val, y_val)
    val_proba = torch_model.predict_proba(X_val)[:, 1]
    best_thr, thr_df = find_best_threshold_v2(y_val, val_proba)
    val_metrics = prediction_metrics_v2(y_val, val_proba, threshold=best_thr)

    return {
        'target': target_name,
        'model_name': 'PyTorch MLP',
        'feature_set': 'engineered',
        'val_threshold': best_thr,
        **{f'val_{k}': v for k, v in val_metrics.items() if k != 'threshold'},
        'best_val_pr_auc_internal': torch_model.best_val_pr_auc_,
        'model_object': torch_model,
        'history': pd.DataFrame(torch_model.history_),
        'threshold_grid': thr_df,
    }

torch_income_v2 = evaluate_pytorch_v2(X_eng_v2, y_income_v2, 'IncomeInvestment')
torch_accum_v2 = evaluate_pytorch_v2(X_eng_v2, y_accum_v2, 'AccumulationInvestment')

display(pd.DataFrame([{
    k: v for k, v in torch_income_v2.items() if k not in {'model_object', 'history', 'threshold_grid'}
}, {
    k: v for k, v in torch_accum_v2.items() if k not in {'model_object', 'history', 'threshold_grid'}
}]))

for title, hist in [('IncomeInvestment', torch_income_v2['history']), ('AccumulationInvestment', torch_accum_v2['history'])]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.lineplot(data=hist, x='epoch', y='train_loss', ax=axes[0])
    axes[0].set_title(f'{title} - Training Loss')
    sns.lineplot(data=hist, x='epoch', y='val_pr_auc', ax=axes[1])
    axes[1].set_title(f'{title} - Validation PR-AUC')
    plt.tight_layout()
    plt.show()


## 6. Final Test-Set Evaluation (Used Once)

At this point the notebook has already:
- explored the data
- engineered features
- benchmarked multiple models
- compared them on validation
- selected final candidates

Now, and only now, do we use the **test set**.

## What the next code does
It compares the best tabular candidate and the PyTorch candidate, then evaluates the final winner on the held-out test set.

## Why this matters
This is the cleanest possible evaluation story:

- **train** → learn patterns
- **validation** → choose model and threshold
- **test** → final one-time performance estimate

If someone asks why the results should be trusted, this section is a big part of the answer.


In [ ]:

def pick_final_config_with_torch(tabular_df, torch_result):
    top_tabular = tabular_df.iloc[0].to_dict()
    tabular_score = (top_tabular['val_pr_auc'], top_tabular['val_f1'], top_tabular['cv_pr_auc_mean'])
    torch_score = (torch_result['val_pr_auc'], torch_result['val_f1'], -np.inf)
    if torch_score > tabular_score:
        return {
            'source': 'torch',
            'feature_set': torch_result['feature_set'],
            'model_name': torch_result['model_name'],
            'val_threshold': torch_result['val_threshold'],
        }
    return {
        'source': 'tabular',
        'feature_set': top_tabular['feature_set'],
        'model_name': top_tabular['model_name'],
        'val_threshold': top_tabular['val_threshold'],
    }

final_pick_income_v2 = pick_final_config_with_torch(benchmark_income_v2, torch_income_v2)
final_pick_accum_v2 = pick_final_config_with_torch(benchmark_accum_v2, torch_accum_v2)

display(pd.DataFrame([final_pick_income_v2, final_pick_accum_v2], index=['IncomeInvestment', 'AccumulationInvestment']))

def bootstrap_ci_v2(y_true, proba, threshold, n_boot=500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    rows = []
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        ys = y_true[idx]
        ps = proba[idx]
        if len(np.unique(ys)) < 2:
            continue
        rows.append(prediction_metrics_v2(ys, ps, threshold))
    boot = pd.DataFrame(rows)
    ci = boot.quantile([0.025, 0.975]).T
    ci.columns = ['ci_low', 'ci_high']
    return ci

def mcnemar_exact_v2(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    a_correct = pred_a == y_true
    b_correct = pred_b == y_true
    n01 = int(np.sum(a_correct & (~b_correct)))
    n10 = int(np.sum((~a_correct) & b_correct))
    p = 1.0 if (n01 + n10) == 0 else binomtest(k=min(n01, n10), n=n01 + n10, p=0.5, alternative='two-sided').pvalue
    return {'baseline_correct_final_wrong': n01, 'baseline_wrong_final_correct': n10, 'mcnemar_p_value': p}

def fit_final_model_v2(target_name, y_series, final_pick):
    feature_frame = feature_sets_v2[final_pick['feature_set']]
    X_train = feature_frame.iloc[train_idx_v2].copy()
    y_train = y_series.iloc[train_idx_v2].copy()
    X_val = feature_frame.iloc[val_idx_v2].copy()
    y_val = y_series.iloc[val_idx_v2].copy()
    X_train_val = feature_frame.iloc[train_val_idx_v2].copy()
    y_train_val = y_series.iloc[train_val_idx_v2].copy()
    X_test = feature_frame.iloc[test_idx_v2].copy()
    y_test = y_series.iloc[test_idx_v2].copy()
    threshold = float(final_pick['val_threshold'])

    # Baseline logistic model
    baseline = build_pipeline_v2('Logistic Regression', build_model_catalog_v2(y_train)['Logistic Regression'])
    baseline.fit(X_train_val, y_train_val)
    base_proba = baseline.predict_proba(X_test)[:, 1]
    base_pred = (base_proba >= 0.50).astype(int)

    if final_pick['source'] == 'torch':
        final_model = TorchMLPBinaryClassifierV2()
        final_model.fit(X_train, y_train, X_val, y_val)  # train with validation supervision for early stopping
        test_proba = final_model.predict_proba(X_test)[:, 1]
        final_pred = (test_proba >= threshold).astype(int)
    else:
        model = build_model_catalog_v2(y_train)[final_pick['model_name']]
        final_model = build_pipeline_v2(final_pick['model_name'], clone(model))
        final_model.fit(X_train_val, y_train_val)
        test_proba = final_model.predict_proba(X_test)[:, 1]
        final_pred = (test_proba >= threshold).astype(int)

    metrics = prediction_metrics_v2(y_test, test_proba, threshold=threshold)
    ci = bootstrap_ci_v2(y_test, test_proba, threshold=threshold)
    mcnemar = mcnemar_exact_v2(y_test, base_pred, final_pred)

    # Plots
    ConfusionMatrixDisplay.from_predictions(y_test, final_pred, cmap='Blues', colorbar=False)
    plt.title(f'Confusion Matrix - {target_name}')
    plt.grid(False)
    plt.show()

    RocCurveDisplay.from_predictions(y_test, test_proba)
    plt.title(f'ROC Curve - {target_name}')
    plt.show()

    p, r, _ = precision_recall_curve(y_test, test_proba)
    plt.figure(figsize=(6, 5))
    plt.plot(r, p)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {target_name}')
    plt.show()

    return {
        'target': target_name,
        'feature_set': final_pick['feature_set'],
        'model_name': final_pick['model_name'],
        'threshold': threshold,
        'model_object': final_model,
        'X_test': X_test,
        'y_test': y_test,
        'test_proba': test_proba,
        'test_pred': final_pred,
        'test_metrics': metrics,
        'bootstrap_ci': ci,
        'mcnemar': mcnemar,
    }

final_income_eval_v2 = fit_final_model_v2('IncomeInvestment', y_income_v2, final_pick_income_v2)
final_accum_eval_v2 = fit_final_model_v2('AccumulationInvestment', y_accum_v2, final_pick_accum_v2)

heldout_metrics_v2 = pd.DataFrame([
    {'target': final_income_eval_v2['target'], 'feature_set': final_income_eval_v2['feature_set'], 'model_name': final_income_eval_v2['model_name'], **final_income_eval_v2['test_metrics']},
    {'target': final_accum_eval_v2['target'], 'feature_set': final_accum_eval_v2['feature_set'], 'model_name': final_accum_eval_v2['model_name'], **final_accum_eval_v2['test_metrics']},
])
display(heldout_metrics_v2)
display(final_income_eval_v2['bootstrap_ci'])
display(final_accum_eval_v2['bootstrap_ci'])
display(pd.DataFrame([final_income_eval_v2['mcnemar'], final_accum_eval_v2['mcnemar']], index=['IncomeInvestment','AccumulationInvestment']))

heldout_metrics_v2.to_csv(OUTPUT_DIR / 'heldout_metrics_v2.csv', index=False)


## 6.1 Diagnostic Plots — ROC, Precision-Recall & Confusion Matrices

Numbers alone are not enough.  
We also need diagnostic visuals.

## What the next code does
It creates standard classification diagnostics for the final selected models:
- ROC curves
- Precision–Recall curves
- confusion matrices

## Why these plots matter
They help answer practical questions such as:
- Is the classifier trading too much precision for recall?
- Are false positives or false negatives dominating?
- Does one target look harder than the other?
- Is the operating threshold sensible?

These plots are produced **after** the model is already selected, so they are used for reporting and understanding — not for secretly re-tuning the pipeline.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ROC, PRECISION-RECALL CURVES & CONFUSION MATRICES
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              confusion_matrix, classification_report)

fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# Store results to avoid refitting
diag_cache = {}

for row, (tname, final_pick, benchmark_df, y_series) in enumerate([
    ('IncomeInvestment', final_pick_income_v2, benchmark_income_v2, y_income_v2),
    ('AccumulationInvestment', final_pick_accum_v2, benchmark_accum_v2, y_accum_v2),
]):
    feat_frame = feature_sets_v2[final_pick['feature_set']]
    X_test = feat_frame.iloc[test_idx_v2].copy()
    y_test = y_series.iloc[test_idx_v2].copy()
    threshold = float(final_pick['val_threshold'])

    # Fit once
    if final_pick['source'] == 'torch':
        deploy = TorchMLPBinaryClassifierV2()
        deploy.fit(
            feat_frame.iloc[train_idx_v2], y_series.iloc[train_idx_v2],
            feat_frame.iloc[val_idx_v2], y_series.iloc[val_idx_v2]
        )
        test_proba = deploy.predict_proba(X_test)[:, 1]
    else:
        model = build_model_catalog_v2(y_series.iloc[train_idx_v2])[final_pick['model_name']]
        pipe = build_pipeline_v2(final_pick['model_name'], clone(model))
        pipe.fit(feat_frame.iloc[train_val_idx_v2], y_series.iloc[train_val_idx_v2])
        test_proba = pipe.predict_proba(X_test)[:, 1]

    test_pred = (test_proba >= threshold).astype(int)
    diag_cache[tname] = {'y_test': y_test, 'test_proba': test_proba,
                         'test_pred': test_pred, 'threshold': threshold,
                         'model_name': final_pick['model_name']}

    # --- ROC Curve ---
    ax_roc = axes[row, 0]
    fpr, tpr, _ = roc_curve(y_test, test_proba)
    auc_val = roc_auc_score(y_test, test_proba)
    ax_roc.plot(fpr, tpr, lw=2, color='#2563eb',
                label=f'{final_pick["model_name"]} (AUC={auc_val:.3f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5)
    ax_roc.fill_between(fpr, tpr, alpha=0.1, color='#2563eb')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title(f'ROC Curve — {tname}')
    ax_roc.legend(loc='lower right')

    # --- Precision-Recall Curve ---
    ax_pr = axes[row, 1]
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, test_proba)
    ap_val = average_precision_score(y_test, test_proba)
    ax_pr.plot(rec_arr, prec_arr, lw=2, color='#dc2626',
               label=f'{final_pick["model_name"]} (AP={ap_val:.3f})')
    ax_pr.fill_between(rec_arr, prec_arr, alpha=0.1, color='#dc2626')
    ax_pr.axhline(y=y_test.mean(), color='gray', ls='--', lw=0.8,
                  label=f'Baseline ({y_test.mean():.2f})')
    ax_pr.set_xlabel('Recall')
    ax_pr.set_ylabel('Precision')
    ax_pr.set_title(f'Precision-Recall Curve — {tname}')
    ax_pr.legend(loc='lower left')

    # --- Confusion Matrix ---
    ax_cm = axes[row, 2]
    cm = confusion_matrix(y_test, test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=['No Need', 'Has Need'],
                yticklabels=['No Need', 'Has Need'],
                annot_kws={'size': 14})
    ax_cm.set_ylabel('Actual')
    ax_cm.set_xlabel('Predicted')
    ax_cm.set_title(f'Confusion Matrix — {tname}\n'
                    f'(threshold={threshold:.2f}, '
                    f'F1={f1_score(y_test, test_pred):.3f})')

plt.suptitle('Final Model Diagnostics — Held-Out Test Set', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Classification reports — reuse cached results, no refit
for tname, cache in diag_cache.items():
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION REPORT — {tname}")
    print(f"Model: {cache['model_name']} | Threshold: {cache['threshold']:.2f}")
    print(f"{'='*60}")
    print(classification_report(cache['y_test'], cache['test_pred'],
          target_names=['No Need (0)', 'Has Need (1)'], digits=4))


## 7. Explainability — Permutation Importance + SHAP

A strong financial ML project should not stop at prediction.  
It should also explain **why** predictions happen.

## What the next code does
It uses:
- **permutation importance** to test how much performance depends on each feature
- **SHAP** to estimate feature contributions in an interpretable way

## Why this is especially useful here
Needs-based recommendation in finance should not be a total black box.  
We want to understand:
- which client characteristics matter most
- whether the model behaves sensibly
- whether the important features match financial intuition

## Special note
If the final selected model is PyTorch, SHAP is still run on the strongest tabular model for stable explanation.
That is a practical compromise:
- keep predictive ambition
- keep explanation stable and readable


In [ ]:

def best_tabular_pick_from_benchmark(benchmark_df):
    row = benchmark_df.iloc[0]
    return {'feature_set': row['feature_set'], 'model_name': row['model_name'], 'val_threshold': row['val_threshold']}

def fit_best_tabular_for_explainability(benchmark_df, y_series):
    pick = best_tabular_pick_from_benchmark(benchmark_df)
    feature_frame = feature_sets_v2[pick['feature_set']]
    X_train_val = feature_frame.iloc[train_val_idx_v2].copy()
    y_train_val = y_series.iloc[train_val_idx_v2].copy()
    model = build_model_catalog_v2(y_train_val)[pick['model_name']]
    pipe = build_pipeline_v2(pick['model_name'], clone(model))
    pipe.fit(X_train_val, y_train_val)
    return pick, pipe, feature_frame.iloc[test_idx_v2].copy(), y_series.iloc[test_idx_v2].copy()

def permutation_importance_v2(pipe, X_test, y_test, title, top_n=12):
    perm = permutation_importance(pipe, X_test, y_test, scoring='average_precision', n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
    imp_df = pd.DataFrame({
        'feature': X_test.columns,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False).head(top_n)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=imp_df, x='importance_mean', y='feature')
    plt.title(f'Permutation Importance - {title}')
    plt.show()
    return imp_df

exp_income_pick_v2, exp_income_pipe_v2, exp_income_Xt_v2, exp_income_yt_v2 = fit_best_tabular_for_explainability(benchmark_income_v2, y_income_v2)
exp_accum_pick_v2, exp_accum_pipe_v2, exp_accum_Xt_v2, exp_accum_yt_v2 = fit_best_tabular_for_explainability(benchmark_accum_v2, y_accum_v2)

perm_income_v2 = permutation_importance_v2(exp_income_pipe_v2, exp_income_Xt_v2, exp_income_yt_v2, 'IncomeInvestment')
perm_accum_v2 = permutation_importance_v2(exp_accum_pipe_v2, exp_accum_Xt_v2, exp_accum_yt_v2, 'AccumulationInvestment')
display(perm_income_v2)
display(perm_accum_v2)

def generic_shap_v2(pipe, X_background, X_explain, title):
    try:
        explainer = shap.Explainer(lambda data: pipe.predict_proba(pd.DataFrame(data, columns=X_background.columns))[:, 1], X_background)
        shap_values = explainer(X_explain)
        plt.figure()
        shap.plots.beeswarm(shap_values, max_display=12, show=False)
        plt.title(f'SHAP Beeswarm - {title}')
        plt.show()
        plt.figure()
        shap.plots.bar(shap_values, max_display=12, show=False)
        plt.title(f'SHAP Global Importance - {title}')
        plt.show()
        return shap_values
    except Exception as e:
        print(f'SHAP failed for {title}:', repr(e))
        return None

_ = generic_shap_v2(
    exp_income_pipe_v2,
    exp_income_Xt_v2.sample(min(200, len(exp_income_Xt_v2)), random_state=RANDOM_STATE),
    exp_income_Xt_v2.sample(min(200, len(exp_income_Xt_v2)), random_state=RANDOM_STATE),
    'IncomeInvestment'
)
_ = generic_shap_v2(
    exp_accum_pipe_v2,
    exp_accum_Xt_v2.sample(min(200, len(exp_accum_Xt_v2)), random_state=RANDOM_STATE),
    exp_accum_Xt_v2.sample(min(200, len(exp_accum_Xt_v2)), random_state=RANDOM_STATE),
    'AccumulationInvestment'
)


## 7.1 Full-Dataset Explainability — Global Feature Understanding

The previous explainability step focuses on the selected evaluation workflow.  
This section goes one step further.

## What the next code does
After model choice is locked, it refits on all available data and computes global explainability measures across the full dataset.

## Why that is reasonable
At this stage:
- model selection is already finished
- we are not tuning anymore
- the goal is not evaluation
- the goal is **the most stable global understanding possible**

So this section helps answer:
> “Across all clients, what are the strongest overall drivers of predicted needs?”

That makes the final explanation broader and more representative than relying only on the test slice.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# FULL-DATASET EXPLAINABILITY — SHAP + PERMUTATION IMPORTANCE
# ═══════════════════════════════════════════════════════════════

def full_dataset_explainability(benchmark_df, y_series, target_name):
    """
    Refit the best tabular model on ALL data, then run
    permutation importance and SHAP on the full dataset.
    """
    pick = best_tabular_pick_from_benchmark(benchmark_df)
    feature_frame = feature_sets_v2[pick['feature_set']].copy()
    y_all = y_series.copy()

    # Fit on entire dataset
    model = build_model_catalog_v2(y_all)[pick['model_name']]
    pipe = build_pipeline_v2(pick['model_name'], clone(model))
    pipe.fit(feature_frame, y_all)

    print(f"\n{'='*60}")
    print(f"FULL-DATASET EXPLAINABILITY — {target_name}")
    print(f"Model: {pick['model_name']} | Features: {pick['feature_set']}")
    print(f"{'='*60}")

    # --- Permutation Importance (full dataset) ---
    print(f"\n  Computing permutation importance on all {len(feature_frame):,} samples...")
    perm = permutation_importance(
        pipe, feature_frame, y_all,
        scoring='average_precision', n_repeats=20,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    imp_df = pd.DataFrame({
        'feature': feature_frame.columns,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.barplot(data=imp_df, x='importance_mean', y='feature', ax=ax,
                xerr=imp_df['importance_std'], palette='viridis')
    ax.set_title(f'Permutation Importance (Full Dataset) — {target_name}')
    ax.set_xlabel('Mean AP Decrease')
    plt.tight_layout()
    plt.show()

    display(imp_df.round(4))

    # --- SHAP (full dataset, subsample for speed) ---
    n_bg = min(500, len(feature_frame))
    n_explain = min(1000, len(feature_frame))
    X_background = feature_frame.sample(n_bg, random_state=RANDOM_STATE)
    X_explain = feature_frame.sample(n_explain, random_state=RANDOM_STATE)

    print(f"\n  Computing SHAP values (background={n_bg}, explain={n_explain})...")
    try:
        explainer = shap.Explainer(
            lambda data: pipe.predict_proba(
                pd.DataFrame(data, columns=feature_frame.columns))[:, 1],
            X_background
        )
        shap_values = explainer(X_explain)

        # Beeswarm — direction + magnitude
        plt.figure(figsize=(10, 7))
        shap.plots.beeswarm(shap_values, max_display=15, show=False)
        plt.title(f'SHAP Beeswarm (Full Dataset) — {target_name}')
        plt.tight_layout()
        plt.show()

        # Bar — global importance
        plt.figure(figsize=(10, 7))
        shap.plots.bar(shap_values, max_display=15, show=False)
        plt.title(f'SHAP Global Importance (Full Dataset) — {target_name}')
        plt.tight_layout()
        plt.show()

        # Waterfall — single client example
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(shap_values[0], show=False)
        plt.title(f'SHAP Waterfall — {target_name} (Sample Client)')
        plt.tight_layout()
        plt.show()

        # Top 2 dependence plots
        top_feats = imp_df['feature'].values[:2]
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        for ax, feat in zip(axes, top_feats):
            feat_idx = list(feature_frame.columns).index(feat)
            shap.plots.scatter(shap_values[:, feat_idx], show=False, ax=ax)
            ax.set_title(f'SHAP Dependence — {feat}')
        plt.suptitle(f'{target_name} — Top Feature Dependence', fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"  SHAP failed: {repr(e)}")

    return pipe, imp_df


# Run for both targets
print("Running full-dataset explainability for both targets...\n")

pipe_income_full, imp_income_full = full_dataset_explainability(
    benchmark_income_v2, y_income_v2, 'IncomeInvestment')

pipe_accum_full, imp_accum_full = full_dataset_explainability(
    benchmark_accum_v2, y_accum_v2, 'AccumulationInvestment')

print("\n✅ Full-dataset explainability complete.")
print("→ These results reflect feature importance across ALL clients, not just the test slice.")
print("→ Use these for presentations and stakeholder communication.")


## 8. Next Best Action (NBA) — Recommendation Engine

This is the business layer of the notebook.

Up to now, the pipeline has answered:
> “What need does the client likely have?”

Now it answers:
> “What should we recommend because of that?”

## What the next code does
It builds an explainable recommendation layer that:
1. predicts both needs
2. matches products to need type
3. checks risk suitability
4. scores and ranks candidates
5. produces top-k recommendation outputs

## Why this is a strong design
The NBA engine is not a mysterious black box.  
It is intentionally interpretable:
- first estimate needs
- then filter the relevant product universe
- then apply business / suitability constraints
- then rank

That mirrors how a real recommendation system in finance should be discussed and defended.


In [ ]:

def refit_deployment_model_v2(target_name, y_series, final_pick):
    feature_frame = feature_sets_v2[final_pick['feature_set']].copy()
    y_all = y_series.copy()

    if final_pick['source'] == 'torch':
        # For deployment, use train/val split for early stopping then score all rows
        deploy_model = TorchMLPBinaryClassifierV2()
        deploy_model.fit(
            feature_frame.iloc[train_idx_v2], y_all.iloc[train_idx_v2],
            feature_frame.iloc[val_idx_v2], y_all.iloc[val_idx_v2]
        )
        all_proba = deploy_model.predict_proba(feature_frame)[:, 1]
    else:
        model = build_model_catalog_v2(y_all.iloc[train_idx_v2])[final_pick['model_name']]
        deploy_model = build_pipeline_v2(final_pick['model_name'], clone(model))
        deploy_model.fit(feature_frame.iloc[train_val_idx_v2], y_all.iloc[train_val_idx_v2])
        all_proba = deploy_model.predict_proba(feature_frame)[:, 1]

    all_pred = (all_proba >= float(final_pick['val_threshold'])).astype(int)

    return {
        'model_object': deploy_model,
        'all_proba': all_proba,
        'all_pred': all_pred,
        'threshold': float(final_pick['val_threshold']),
        'feature_set': final_pick['feature_set'],
    }

deploy_income_v2 = refit_deployment_model_v2('IncomeInvestment', y_income_v2, final_pick_income_v2)
deploy_accum_v2 = refit_deployment_model_v2('AccumulationInvestment', y_accum_v2, final_pick_accum_v2)

nba_clients_v2 = df_final.copy()
nba_clients_v2['IncomeNeedProbability'] = deploy_income_v2['all_proba']
nba_clients_v2['IncomeNeedPredicted'] = deploy_income_v2['all_pred']
nba_clients_v2['AccumulationNeedProbability'] = deploy_accum_v2['all_proba']
nba_clients_v2['AccumulationNeedPredicted'] = deploy_accum_v2['all_pred']

product_catalog_v2 = products_df.copy()
if 'Name' not in product_catalog_v2.columns:
    product_catalog_v2['Name'] = product_catalog_v2['IDProduct'].astype(str)
if 'TypeLabel' not in product_catalog_v2.columns:
    product_catalog_v2['TypeLabel'] = product_catalog_v2['Type'].map({0: 'Income', 1: 'Accumulation'})

def rank_products_for_need_v2(client_row, need_name, prob, top_k=3):
    type_code = 0 if need_name == 'IncomeInvestment' else 1
    eligible = product_catalog_v2[product_catalog_v2['Type'] == type_code].copy()

    # Risk suitability: do not exceed client risk propensity
    eligible = eligible[eligible['Risk'] <= client_row['RiskPropensity']].copy()
    if eligible.empty:
        return pd.DataFrame(columns=['IDProduct', 'Name', 'TypeLabel', 'Risk', 'NeedProbability', 'RiskMatchScore', 'SuitabilityScore'])

    risk_gap = (client_row['RiskPropensity'] - eligible['Risk']).abs()
    eligible['NeedProbability'] = prob
    eligible['RiskMatchScore'] = 1.0 - (risk_gap / max(client_row['RiskPropensity'], 1))
    eligible['SuitabilityScore'] = 0.70 * eligible['NeedProbability'] + 0.30 * eligible['RiskMatchScore']

    # Conservative age adjustment
    if client_row['Age'] >= 65:
        eligible.loc[eligible['Risk'] > eligible['Risk'].median(), 'SuitabilityScore'] -= 0.05

    return eligible.sort_values(['SuitabilityScore', 'Risk'], ascending=[False, False]).head(top_k)

nba_rows_v2 = []

for idx, client in nba_clients_v2.iterrows():
    income_prob = float(client['IncomeNeedProbability'])
    accum_prob = float(client['AccumulationNeedProbability'])
    income_pred = int(client['IncomeNeedPredicted'])
    accum_pred = int(client['AccumulationNeedPredicted'])

    income_ranked = rank_products_for_need_v2(client, 'IncomeInvestment', income_prob, top_k=3) if income_pred == 1 else pd.DataFrame()
    accum_ranked = rank_products_for_need_v2(client, 'AccumulationInvestment', accum_prob, top_k=3) if accum_pred == 1 else pd.DataFrame()

    if income_pred == 1 and accum_pred == 1:
        primary_need = 'AccumulationInvestment' if accum_prob >= income_prob else 'IncomeInvestment'
    elif income_pred == 1:
        primary_need = 'IncomeInvestment'
    elif accum_pred == 1:
        primary_need = 'AccumulationInvestment'
    else:
        primary_need = 'No predicted need'

    primary_ranked = accum_ranked if primary_need == 'AccumulationInvestment' else income_ranked if primary_need == 'IncomeInvestment' else pd.DataFrame()
    top_primary = primary_ranked.iloc[0].to_dict() if not primary_ranked.empty else {}

    nba_rows_v2.append({
        'ClientID': client.get('ID', idx),
        'Age': client['Age'],
        'Gender': client['Gender'],
        'FamilyMembers': client['FamilyMembers'],
        'FinancialEducation': client['FinancialEducation'],
        'RiskPropensity': client['RiskPropensity'],
        'Income': client['Income'],
        'Wealth': client['Wealth'],
        'IncomeNeedProbability': income_prob,
        'IncomeNeedPredicted': income_pred,
        'AccumulationNeedProbability': accum_prob,
        'AccumulationNeedPredicted': accum_pred,
        'PrimaryNeed': primary_need,
        'PrimaryRecommendedProductID': top_primary.get('IDProduct', np.nan),
        'PrimaryRecommendedProductName': top_primary.get('Name', np.nan),
        'PrimaryRecommendedProductRisk': top_primary.get('Risk', np.nan),
        'PrimaryRecommendationScore': top_primary.get('SuitabilityScore', np.nan),
        'IncomeTop3': ' | '.join(income_ranked['Name'].astype(str).tolist()) if not income_ranked.empty else np.nan,
        'AccumulationTop3': ' | '.join(accum_ranked['Name'].astype(str).tolist()) if not accum_ranked.empty else np.nan,
    })

nba_output_v2 = pd.DataFrame(nba_rows_v2)
display(nba_output_v2.head(15))

coverage_summary_v2 = pd.DataFrame({
    'metric': [
        'Predicted income need',
        'Predicted accumulation need',
        'At least one predicted need',
        'Primary recommendation available',
    ],
    'value': [
        int(nba_output_v2['IncomeNeedPredicted'].sum()),
        int(nba_output_v2['AccumulationNeedPredicted'].sum()),
        int(((nba_output_v2['IncomeNeedPredicted'] == 1) | (nba_output_v2['AccumulationNeedPredicted'] == 1)).sum()),
        int(nba_output_v2['PrimaryRecommendedProductID'].notna().sum()),
    ]
})
display(coverage_summary_v2)

top_primary_products_v2 = nba_output_v2['PrimaryRecommendedProductName'].dropna().value_counts().rename_axis('Product').reset_index(name='Count')
if not top_primary_products_v2.empty:
    plt.figure(figsize=(9, 5))
    sns.barplot(data=top_primary_products_v2, x='Count', y='Product')
    plt.title('Primary NBA Recommendations')
    plt.show()

nba_output_v2.to_csv(OUTPUT_DIR / 'nba_output_v2.csv', index=False)
coverage_summary_v2.to_csv(OUTPUT_DIR / 'nba_coverage_summary_v2.csv', index=False)


## 9. Summary & Conclusions

This notebook is strong because it combines **ambition** with **methodological discipline**.

## What was done
- rich exploratory analysis
- statistical testing
- careful feature engineering
- multiple tabular ML models
- PyTorch deep learning benchmark
- validation-based threshold tuning
- one-time final test evaluation
- explainability via permutation importance and SHAP
- business translation into an NBA recommendation engine

## Why the final workflow is defendable
The strongest methodological point is the separation:

- **training data** for learning and cross-validation
- **validation data** for threshold tuning and final candidate choice
- **test data** for a single locked final evaluation

## Final one-line explanation
If you need to explain the entire notebook very simply, say:

> “We first understood the client data, then built leakage-safe ML and DL models to predict financial needs, then explained those predictions, and finally turned them into risk-aware next best action recommendations.”

That is the full story of the notebook in one sentence.
